In [ ]:
import numpy as np
import pandas as pd
from wfdb import rdsamp
from pathlib import Path
from tqdm import tqdm
from _multilabel_stratified_sampling import stratify

dataset_path = Path("/opt/gpudata/ecg/zzu-pecg")

### Prepare Labels

In [ ]:
fine_grained_labels = {
    "Fulminant myocarditis": ["(F) I40.0"],
    "Viral myocarditis": ["(V) I40.0"],
    "Acute myocarditis": ["I40.9"],
    "_Myocarditis": ["I51.4"],
    "Dilated cardiomypoathy": ["I42.0"],
    "Hypertrophic cardiomyopathy": ["I42.2"],
    "_Cardiomyopathy": ["I42.9"],
    "Noncompaction of the ventricular myocardium": ["Q24.8"],
    "_Kawasaki disease": ["M30.3"],
    "Ventricular septal defect": ["Q21.0"],
    "Atrial septal defect": ["Q21.1"],
    "Atrial septal defect (Foramen ovale)": ["(FO) Q21.1"],
    "Atrial septal defect (Ostium secundum defect)": ["(OSD) Q21.1"],
    "Atrioventricular septal defect": ["Q21.2"],
    "Tetralogy of Fallot": ["Q21.3"],
    "Stenosis of right ventricular outflow tract": ["Q22.1"],
    "Patent ductus arteriosus": ["Q25.0"],
    "Pulmonary stenosis": ["Q25.6"],
    "Pulmonary valve stenosis": ["I37.0"],
}

In [ ]:
df = pd.read_csv(dataset_path / "AttributesDictionary.csv")
df = df[df["Lead"] == 12].sort_values(["Patient_ID", "ECG_ID"]).reset_index(drop=True)
original_cols = df.columns

In [ ]:
# one-hot encode labels
dxs = df["ICD-10 code"].str.split(";").explode() # create a long table of idx --> icd code, where idx can appear multiple time
dx_idxs = {icd.strip("'"): list(group.index) for icd, group in dxs.groupby(dxs)} # convert to dict of icd code --> list[idx]

# convert to wide
for label, icds in fine_grained_labels.items():
    df[label] = 0
    for icd in icds:
        df.loc[dx_idxs[icd], label] = 1

### Create Initial Splits

In [ ]:
pt_labels = df.groupby("Patient_ID")[list(fine_grained_labels)].max()
assert df["Age"].str.endswith("d").all()
df["_age"] = df["Age"].str[:-1].astype(int)
mean_pt_age = df.groupby("Patient_ID")["_age"].mean()
pt_sex = df.groupby("Patient_ID")["Gender"].first()
binned_age = pd.qcut(mean_pt_age, q=4)
one_hot_age = pd.get_dummies(binned_age).astype(int)
one_hot_sex = pd.get_dummies(pt_sex).astype(int)
stratifier = pd.concat([one_hot_sex, one_hot_age, pt_labels], axis=1)

num_ecgs_per_pt = df.groupby("Patient_ID").size()
assert (num_ecgs_per_pt.index == pt_labels.index).all()
assert (num_ecgs_per_pt.index == one_hot_age.index).all()
assert (num_ecgs_per_pt.index == one_hot_sex.index).all()
assert (num_ecgs_per_pt.index == stratifier.index).all()
pts = pt_labels.index

In [ ]:
n_patients, n_classes = stratifier.shape

label_lists = []
for row in stratifier.to_numpy():
    label_idxs = np.where(row)[0].tolist()
    if len(label_idxs) == 0:
        label_idxs = [n_classes] # no-label fallback, stratifier needs non-empty labels
    label_lists.append(label_idxs)

stratified_ids, stratified_labels = stratify(
    data=label_lists,
    classes=list(range(n_classes+1)),
    ratios=[0.7, 0.1, 0.2],
    qualities=[2] * n_patients, # no notion of quality
    ecgs_per_patient=num_ecgs_per_pt.to_list(),
    nr_clean_folds=0,
    random_seed=42, # found a random seed that ensures label presence in val/test - which is further complicated by first ECG requirement
)

assert n_patients == sum(len(x) for x in stratified_ids)

In [ ]:
dummy_mask = np.zeros(len(df)).astype(bool) # all False
train_mask = dummy_mask.copy()
val_mask = dummy_mask.copy()
test_mask = dummy_mask.copy()

# also require first ECG per patient in val/test
assert (df.sort_values(["Patient_ID", "ECG_ID"]).index == df.index).all()
first_ecg_mask = ~df.duplicated("Patient_ID", keep="first")

for i, pt_idxs in enumerate(stratified_ids):
    fold_pts = pts[pt_idxs]
    fold_idxs = df[df["Patient_ID"].isin(fold_pts)].index.to_numpy()
    if i == 1:
        val_mask[fold_idxs] = True
        val_mask &= first_ecg_mask
    elif i == 2:
        test_mask[fold_idxs] = True
        test_mask &= first_ecg_mask
    else:
        train_mask[fold_idxs] = True

In [ ]:
# check that all fine grained labels present in train/val/test
assert (df.loc[train_mask, list(fine_grained_labels)].sum() > 0).all()
assert (df.loc[val_mask, list(fine_grained_labels)].sum() > 0).all()
assert (df.loc[test_mask, list(fine_grained_labels)].sum() > 0).all()

df["split"] = "no_split"
df.loc[train_mask, "split"] = "train"
df.loc[val_mask, "split"] = "val"
df.loc[test_mask, "split"] = "test"

In [ ]:
df["split"].value_counts()

In [ ]:
save_cols = ["Patient_ID", "ECG_ID", "split", "Filename", "Age", "Gender", "Acquisition_date", "Sampling_point", "Lead"] + list(fine_grained_labels)
df.loc[train_mask | val_mask | test_mask, save_cols].to_csv(dataset_path / "labels.csv", index=False)

### For Defines

In [ ]:
list(fine_grained_labels)

In [ ]:
# compute waveform stats:
# because ZZU ECGs are variable length, can't stack directly
# however, stats are computed along time dim as well, so we can
# just concenate all ECGs end to end before computing stats
train_df = df[df["split"] == "train"].reset_index(drop=True)
train_df["fpath"] = dataset_path / "Child_ecg" / train_df["Filename"]
X = []
for i, fpath in enumerate(tqdm(train_df["fpath"])):
    x, _ = rdsamp(fpath)
    n_timesteps, n_leads = x.shape
    assert n_leads == 12
    assert train_df.loc[i, "Sampling_point"] == n_timesteps
    assert not np.isnan(x).any()
    X.append(x)
X = np.concatenate(X)

In [ ]:
lowers, uppers = np.percentile(X, [0.1, 99.9], axis=0)
display(lowers.tolist())
display(uppers.tolist())

In [ ]:
X_clipped = np.clip(X, lowers, uppers)
means = X_clipped.mean(axis=0)
stds = X_clipped.std(axis=0)
display(means.tolist())
display(stds.tolist())